# 🇹🇷 Türkçe Bütünleşik Artgönderim Çözümleme Sistemi
## v3.0 - Geliştirilmiş Gizli Özne Tespiti

**Özellikler:**
- ✅ Zamir Artgönderimi (O, Bu, Şu, Onlar...)
- ✅ Gizli Özne / Boş Artgönderim Tespiti
- ✅ Açıklayıcı Türkçe Çıktılar
- ✅ Gradio Arayüzü

---

## 1. Kurulum

In [ ]:
!pip install -q transformers torch gradio pandas numpy
!pip install -q sentencepiece accelerate

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForTokenClassification
import gradio as gr
import pandas as pd
import numpy as np
import json
import re
import os
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Cihaz: {device}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")

 Cihaz: cuda
 GPU: Tesla T4


## 2. Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Models"

MODEL_PATHS = {
    'pronoun_bert': f"{BASE_DIR}/pronoun_ensemble/bert-base-turkish-cased",
    'pronoun_deberta': f"{BASE_DIR}/pronoun_ensemble/deberta-v3-small",
    'zero_bert': f"/content/drive/MyDrive/ModelZAR/bertzartr/checkpoint-585",
}

print("📂 Model Durumu:")
for name, path in MODEL_PATHS.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"   {exists} {name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Model Durumu:
   ✅ pronoun_bert
   ✅ pronoun_deberta
   ✅ zero_bert


## 3. Veri Yapıları ve Kurallar

In [ ]:
class AnaphoraType(Enum):
    PRONOUN = "zamir"
    ZERO = "gizli_özne"


@dataclass
class Finding:
    """Tek bir bulgu"""
    sentence_id: int
    sentence: str
    finding_type: AnaphoraType
    trigger_word: str
    antecedent: str
    confidence: float

    def explain(self) -> str:
        """Türkçe açıklama"""
        if self.finding_type == AnaphoraType.PRONOUN:
            return (f'"{self.sentence}" cümlesinde '
                    f"'{self.trigger_word}' zamiri '{self.antecedent}' kelimesine atıf yapmaktadır.")
        return (f'"{self.sentence}" cümlesinde gizli özne (boş artgönderim) vardır. '
                f"Öncül kelime '{self.antecedent}' olarak tespit edilmiştir.")


# Ortak BIO etiket şeması (zero modeli farklı isim kullansa da normalize edilir)
DEFAULT_BIO_SCHEMA = {
    "O": 0,
    "B-PRONOUN": 1,
    "I-PRONOUN": 2,
    "B-ZERO": 3,
    "I-ZERO": 4,
}


class TurkishRules:
    """Türkçe dilbilgisi kuralları"""

    SINGULAR_PRONOUNS = ['o', 'bu', 'şu', 'kendisi', 'kendi', 'onu', 'ona', 'onun']
    PLURAL_PRONOUNS = ['onlar', 'bunlar', 'şunlar', 'kendileri', 'onları', 'onlara']
    ALL_PRONOUNS = SINGULAR_PRONOUNS + PLURAL_PRONOUNS

    NON_SUBJECT_WORDS = [
        'hemen', 'sonra', 'önce', 'şimdi', 'dün', 'bugün', 'yarın', 'artık', 'henüz',
        'hâlâ', 'yine', 'tekrar', 'derhal', 'aniden', 'birden', 'hızla', 'yavaşça',
        'çok', 'az', 'biraz', 'pek', 'gayet', 'oldukça', 'fazla', 'daha', 'en',
        'ama', 'fakat', 'ancak', 'lakin', 've', 'veya', 'ya', 'yahut', 'ile',
        'çünkü', 'zira', 'madem', 'eğer', 'şayet', 'oysa', 'halbuki',
        'için', 'gibi', 'kadar', 'göre', 'karşı', 'rağmen', 'doğru',
        'böylece', 'dolayısıyla', 'ayrıca', 'üstelik', 'dahası', 'özellikle',
        'genellikle', 'bazen', 'nadiren', 'asla', 'hiç', 'belki', 'muhtemelen',
        'sonucu', 'sonucunda', 'ardından', 'akabinde', 'nedeniyle', 'sayesinde',
        'neden', 'niçin', 'nasıl', 'nerede', 'ne', 'kim', 'hangi', 'kaç',
        'gelerek', 'gidip', 'alıp', 'yapıp', 'olup', 'edip', 'görüp', 'duyup',
        'öğrenmek', 'anlamak', 'bilmek', 'görmek', 'duymak', 'almak', 'vermek'
    ]

    PLURAL_VERB_SUFFIXES = [
        'dılar', 'diler', 'dular', 'düler', 'tılar', 'tiler', 'tular', 'tüler',
        'mışlar', 'mişler', 'muşlar', 'müşler', 'yorlar', 'arlar', 'erler',
        'ırlar', 'irler', 'urlar', 'ürler', 'acaklar', 'ecekler',
        'lardı', 'lerdi', 'larmış', 'lermiş'
    ]

    SINGULAR_VERB_SUFFIXES = [
        'dı', 'di', 'du', 'dü', 'tı', 'ti', 'tu', 'tü',
        'mış', 'miş', 'muş', 'müş', 'yor', 'ar', 'er',
        'ır', 'ir', 'ur', 'ür', 'acak', 'ecek', 'malı', 'meli'
    ]

    @classmethod
    def is_pronoun(cls, word: str) -> bool:
        return word.lower().strip('.,!?;:') in cls.ALL_PRONOUNS

    @classmethod
    def is_plural_pronoun(cls, word: str) -> bool:
        return word.lower().strip('.,!?;:') in cls.PLURAL_PRONOUNS

    @classmethod
    def has_explicit_subject(cls, tokens: List[str]) -> bool:
        if not tokens:
            return False
        first = tokens[0].strip('.,!?;:')
        first_lower = first.lower()
        if first_lower in cls.NON_SUBJECT_WORDS:
            return False
        if first and first[0].islower():
            return False
        if cls._is_verb(first_lower):
            return False
        return bool(first and first[0].isupper())

    @classmethod
    def _is_verb(cls, word: str) -> bool:
        word = word.lower().strip('.,!?;:')
        for suffix in cls.SINGULAR_VERB_SUFFIXES + cls.PLURAL_VERB_SUFFIXES:
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                return True
        return False

    @classmethod
    def get_verb_plurality(cls, tokens: List[str]) -> Optional[bool]:
        for token in tokens:
            word = token.lower().strip('.,!?;:')
            for suffix in cls.PLURAL_VERB_SUFFIXES:
                if word.endswith(suffix):
                    return True
            for suffix in cls.SINGULAR_VERB_SUFFIXES:
                if word.endswith(suffix) and len(word) > len(suffix) + 2:
                    return False
        return None


print("✅ Veri yapıları ve kurallar yüklendi")
print("📌 BIO şeması:", DEFAULT_BIO_SCHEMA)



✅ Veri yapıları ve kurallar yüklendi
   'Hemen uyudu' → Özne var mı? False
   'Ali geldi' → Özne var mı? True
   'Sonucu öğrenmek' → Özne var mı? False


## 4. Ana Çözümleyici Sınıf

In [ ]:
class TurkishAnaphoraResolver:
    """
    Türkçe Artgönderim Çözümleyici (Hibrit)
    - BERT ile zamir + zero adayı tespiti
    - GPT ile tek prompt'ta zamir/zero + antecedent çözümleme
    - Kural tabanı sadece son çare (fallback)
    """

    def __init__(self, pronoun_model_path: str = None, zero_model_path: str = None, gpt_model: str = "gpt-4o-mini"):
        print("=" * 60)
        print("🚀 Türkçe Artgönderim Çözümleyici")
        print("=" * 60)

        self.pronoun_model = None
        self.zero_model = None
        self.gpt_model = gpt_model
        self.gpt_client = None

        if pronoun_model_path and os.path.exists(pronoun_model_path):
            self._load_pronoun_model(pronoun_model_path)

        if zero_model_path and os.path.exists(zero_model_path):
            self._load_zero_model(zero_model_path)

        self._init_gpt_client()

        print("\n✅ Sistem hazır!")
        print("=" * 60)

    def _init_gpt_client(self):
        try:
            from openai import OpenAI
            if os.getenv("OPENAI_API_KEY"):
                self.gpt_client = OpenAI()
                print("🤖 GPT istemcisi aktif")
            else:
                print("⚠️ OPENAI_API_KEY yok, GPT adımı fallback modunda")
        except Exception as e:
            print(f"⚠️ OpenAI istemcisi yüklenemedi: {e}")

    def _load_pronoun_model(self, path):
        print("🔤 Zamir modeli yükleniyor...")
        try:
            self.pronoun_tokenizer = AutoTokenizer.from_pretrained(path)
            self.pronoun_model = AutoModelForTokenClassification.from_pretrained(path).to(device)
            self.pronoun_model.eval()
            print("   ✅ Yüklendi")
        except Exception as e:
            print(f"   ⚠️ Hata: {e}")

    def _load_zero_model(self, path):
        print("👻 Gizli özne modeli yükleniyor...")
        try:
            self.zero_tokenizer = AutoTokenizer.from_pretrained(path)
            self.zero_model = AutoModelForTokenClassification.from_pretrained(path).to(device)
            self.zero_model.eval()
            self.zero_id2label = getattr(self.zero_model.config, "id2label", {})
            print("   ✅ Yüklendi")
            print(f"   🏷️ Zero model etiketleri: {self.zero_id2label}")
        except Exception as e:
            print(f"   ⚠️ Hata: {e}")

    def analyze(self, text: str) -> Dict:
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        findings = []

        for sent_id, sentence in enumerate(sentences):
            tokens = sentence.split()
            gpt_items = self._gpt_detect_and_resolve(sentences, sent_id)

            if gpt_items:
                findings.extend(gpt_items)
                continue

            findings.extend(self._find_pronouns(sentences, sent_id, tokens))

            # Önceki kısıt kaldırıldı: her cümlede zero aranır
            zero_finding = self._check_zero_anaphora(sentences, sent_id, tokens)
            if zero_finding:
                findings.append(zero_finding)

        return {
            'text': text,
            'sentences': sentences,
            'findings': findings,
            'pronoun_count': sum(1 for f in findings if f.finding_type == AnaphoraType.PRONOUN),
            'zero_count': sum(1 for f in findings if f.finding_type == AnaphoraType.ZERO)
        }

    def _predict_token_labels(self, tokens: List[str], model, tokenizer) -> List[Tuple[str, str, float]]:
        if model is None or tokenizer is None or not tokens:
            return []

        enc = tokenizer(tokens, is_split_into_words=True, return_tensors='pt', truncation=True).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)
            pred_ids = probs.argmax(dim=-1).squeeze(0).cpu().tolist()
            pred_conf = probs.max(dim=-1).values.squeeze(0).cpu().tolist()

        word_ids = enc.word_ids(batch_index=0)
        seen = set()
        out = []
        id2label = getattr(model.config, 'id2label', {})

        for tok_idx, word_idx in enumerate(word_ids):
            if word_idx is None or word_idx in seen:
                continue
            seen.add(word_idx)
            label = id2label.get(int(pred_ids[tok_idx]), str(pred_ids[tok_idx]))
            out.append((tokens[word_idx], label.upper(), float(pred_conf[tok_idx])))
        return out

    def _gpt_detect_and_resolve(self, sentences: List[str], sent_id: int) -> List[Finding]:
        if self.gpt_client is None:
            return []

        context = " ".join(sentences[max(0, sent_id - 2):sent_id])
        sentence = sentences[sent_id]

        prompt = f"""
Aşağıdaki Türkçe cümlede artgönderimleri bul.
- Hem zamirleri hem gizli özne (zero anaphora) adaylarını tespit et.
- Her aday için antecedent belirt.
- Sadece JSON döndür.

JSON şeması:
{{"items": [{{"type": "pronoun|zero", "trigger": "...", "antecedent": "...", "confidence": 0.0}}]}}

Önceki bağlam: {context if context else '[YOK]'}
Hedef cümle: {sentence}
""".strip()

        try:
            response = self.gpt_client.responses.create(
                model=self.gpt_model,
                input=prompt,
                temperature=0.0
            )
            raw = response.output_text
            data = json.loads(raw)
        except Exception:
            return []

        findings = []
        for item in data.get("items", []):
            typ = item.get("type", "").lower()
            antecedent = item.get("antecedent", "").strip()
            trigger = item.get("trigger", "").strip() or "[GİZLİ ÖZNE]"
            conf = float(item.get("confidence", 0.75))

            if not antecedent:
                continue

            if typ == "pronoun":
                findings.append(Finding(sent_id, sentence, AnaphoraType.PRONOUN, trigger, antecedent, conf))
            elif typ == "zero":
                findings.append(Finding(sent_id, sentence, AnaphoraType.ZERO, '[GİZLİ ÖZNE]', antecedent, conf))
        return findings

    def _find_pronouns(self, sentences: List[str], sent_id: int, tokens: List[str]) -> List[Finding]:
        findings = []

        bert_preds = self._predict_token_labels(tokens, self.pronoun_model, getattr(self, 'pronoun_tokenizer', None))
        bert_pronouns = {tok for tok, label, _ in bert_preds if 'PRON' in label}

        for idx, token in enumerate(tokens):
            if token in bert_pronouns or TurkishRules.is_pronoun(token):
                antecedent = self._find_antecedent(
                    sentences, sent_id, tokens, idx,
                    is_plural=TurkishRules.is_plural_pronoun(token)
                )
                if antecedent:
                    findings.append(Finding(sent_id, sentences[sent_id], AnaphoraType.PRONOUN, token, antecedent, 0.95))
        return findings

    def _check_zero_anaphora(self, sentences: List[str], sent_id: int, tokens: List[str]) -> Optional[Finding]:
        bert_preds = self._predict_token_labels(tokens, self.zero_model, getattr(self, 'zero_tokenizer', None))

        zero_detected = any(
            ('ZERO' in label or 'ANAPHOR' in label or 'GAP' in label) and not label.endswith('O')
            for _, label, _ in bert_preds
        )

        # BERT yoksa veya etiketle yakalayamazsa, son çare kural
        if not zero_detected:
            zero_detected = not TurkishRules.has_explicit_subject(tokens)

        if zero_detected:
            is_plural = TurkishRules.get_verb_plurality(tokens)
            antecedent = self._find_antecedent(sentences, sent_id, tokens, -1, is_plural)
            if antecedent:
                return Finding(sent_id, sentences[sent_id], AnaphoraType.ZERO, '[GİZLİ ÖZNE]', antecedent, 0.92)
        return None

    def _find_antecedent(self, sentences: List[str], sent_id: int,
                         tokens: List[str], trigger_idx: int,
                         is_plural: Optional[bool] = None) -> Optional[str]:
        if trigger_idx > 0:
            for i in range(trigger_idx - 1, -1, -1):
                word = tokens[i].strip('.,!?;:')
                if self._is_valid_antecedent(word, is_plural):
                    return word

        for prev_id in range(sent_id - 1, -1, -1):
            prev_sent = sentences[prev_id]
            prev_tokens = prev_sent.split()

            if ' ve ' in prev_sent:
                match = re.search(r'([A-ZÇĞİÖŞÜ][a-zçğıöşü]+)\s+ve\s+([A-ZÇĞİÖŞÜ][a-zçğıöşü]+)', prev_sent)
                if match and (is_plural is None or is_plural):
                    return f"{match.group(1)} ve {match.group(2)}"

            for token in prev_tokens:
                word = token.strip('.,!?;:')
                if self._is_valid_antecedent(word, is_plural):
                    return word
        return None

    def _is_valid_antecedent(self, word: str, is_plural: Optional[bool]) -> bool:
        if not word or not word[0].isupper():
            return False

        word_lower = word.lower()
        if word_lower in TurkishRules.ALL_PRONOUNS or word_lower in TurkishRules.NON_SUBJECT_WORDS:
            return False

        if is_plural is not None:
            word_is_plural = word_lower.endswith(('lar', 'ler'))
            return is_plural == word_is_plural

        return True

    def build_integrated_eval_set(self, dpa_df: pd.DataFrame, zartr_df: pd.DataFrame, per_dataset: int = 50,
                                  seed: int = 42) -> pd.DataFrame:
        """DPATransLLM + ZARTR karışık 100 cümlelik değerlendirme seti üretir."""
        dpa = dpa_df.sample(min(per_dataset, len(dpa_df)), random_state=seed).copy()
        zartr = zartr_df.sample(min(per_dataset, len(zartr_df)), random_state=seed).copy()
        dpa['source'] = 'DPATransLLM'
        zartr['source'] = 'ZARTR'
        combined = pd.concat([dpa, zartr], ignore_index=True)
        return combined.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    def evaluate(self, eval_df: pd.DataFrame,
                 text_col: str = 'sentence', pronoun_gold_col: str = 'pronoun_gold',
                 zero_gold_col: str = 'zero_gold', combined_gold_col: str = 'combined_gold') -> Dict:
        """Pronoun Accuracy + Zero Detection F1 + Combined Accuracy"""
        pronoun_correct = 0
        tp = fp = fn = 0
        combined_correct = 0

        for _, row in eval_df.iterrows():
            result = self.analyze(str(row[text_col]))
            findings = result['findings']

            pred_pron = [f.antecedent for f in findings if f.finding_type == AnaphoraType.PRONOUN]
            pred_zero = [f.antecedent for f in findings if f.finding_type == AnaphoraType.ZERO]

            if str(row.get(pronoun_gold_col, '')) in pred_pron:
                pronoun_correct += 1

            zero_gold = int(row.get(zero_gold_col, 0))
            zero_pred = int(len(pred_zero) > 0)
            if zero_gold == 1 and zero_pred == 1:
                tp += 1
            elif zero_gold == 0 and zero_pred == 1:
                fp += 1
            elif zero_gold == 1 and zero_pred == 0:
                fn += 1

            if str(row.get(combined_gold_col, '')) in [f.antecedent for f in findings]:
                combined_correct += 1

        n = max(len(eval_df), 1)
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = (2 * precision * recall) / max(precision + recall, 1e-9)

        return {
            'sample_size': len(eval_df),
            'pronoun_accuracy': pronoun_correct / n,
            'zero_detection_f1': f1,
            'combined_accuracy': combined_correct / n
        }

    def format_output(self, result: Dict) -> str:
        lines = []
        lines.append("=" * 70)
        lines.append("📊 ARTGÖNDERIM ANALİZ RAPORU")
        lines.append("=" * 70)
        lines.append(f"\n📝 Girdi Metni:\n   \"{result['text']}\"")
        lines.append(f"\n📋 Cümleler ({len(result['sentences'])} adet):")
        for i, sent in enumerate(result['sentences']):
            lines.append(f"   [{i + 1}] {sent}")

        lines.append("\n📈 Özet:")
        lines.append(f"   • Zamir artgönderimi: {result['pronoun_count']} adet")
        lines.append(f"   • Gizli özne: {result['zero_count']} adet")

        if result['findings']:
            lines.append("\n" + "=" * 70)
            lines.append("🔍 DETAYLI BULGULAR")
            lines.append("=" * 70)
            for f in result['findings']:
                icon = "🔤" if f.finding_type == AnaphoraType.PRONOUN else "👻"
                lines.append(f"\n{icon} {f.explain()}")
        else:
            lines.append("\n⚠️ Artgönderim bulunamadı.")

        lines.append("\n" + "=" * 70)
        return "\n".join(lines)


print("✅ TurkishAnaphoraResolver tanımlandı (BERT + GPT + Eval)")



✅ TurkishAnaphoraResolver tanımlandı


## 5. Sistemi Başlat

In [ ]:
resolver = TurkishAnaphoraResolver(
    pronoun_model_path=MODEL_PATHS.get('pronoun_bert'),
    zero_model_path=MODEL_PATHS.get('zero_bert')
)

🚀 Türkçe Artgönderim Çözümleyici
🔤 Zamir modeli yükleniyor...
   ✅ Yüklendi
👻 Gizli özne modeli yükleniyor...
   ✅ Yüklendi

✅ Sistem hazır!


## 6. Test

In [ ]:
# TEST 1
print("\n" + "#"*70)
print("TEST 1: Ali örneği")
print("#"*70)

test1 = "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."
result1 = resolver.analyze(test1)
print(resolver.format_output(result1))


######################################################################
TEST 1: Ali örneği
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."

📋 Cümleler (3 adet):
   [1] Ali dün eve geç geldi
   [2] O çok yorgundu
   [3] Hemen uyudu

📈 Özet:
   • Zamir artgönderimi: 1 adet
   • Gizli özne: 1 adet

🔍 DETAYLI BULGULAR

🔤 "O çok yorgundu" cümlesinde 'O' zamiri 'Ali' kelimesine atıf yapmaktadır.

👻 "Hemen uyudu" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Ali' olarak tespit edilmiştir.



In [ ]:
# TEST 2
print("\n" + "#"*70)
print("TEST 2: Merve ve Cem örneği")
print("#"*70)

test2 = "Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."
result2 = resolver.analyze(test2)
print(resolver.format_output(result2))


######################################################################
TEST 2: Merve ve Cem örneği
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."

📋 Cümleler (2 adet):
   [1] Merve ve Cem hemen odaya girdiler
   [2] Sonucu öğrenmek için bilgisayarı açtılar

📈 Özet:
   • Zamir artgönderimi: 0 adet
   • Gizli özne: 1 adet

🔍 DETAYLI BULGULAR

👻 "Sonucu öğrenmek için bilgisayarı açtılar" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Merve ve Cem' olarak tespit edilmiştir.



In [ ]:
# TEST 3: Birleşik
print("\n" + "#"*70)
print("TEST 3: Birleşik metin")
print("#"*70)

test3 = """Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu.
Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."""

result3 = resolver.analyze(test3)
print(resolver.format_output(result3))


######################################################################
TEST 3: Birleşik metin
######################################################################
📊 ARTGÖNDERIM ANALİZ RAPORU

📝 Girdi Metni:
   "Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu.
Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."

📋 Cümleler (5 adet):
   [1] Ali dün eve geç geldi
   [2] O çok yorgundu
   [3] Hemen uyudu
   [4] Merve ve Cem hemen odaya girdiler
   [5] Sonucu öğrenmek için bilgisayarı açtılar

📈 Özet:
   • Zamir artgönderimi: 1 adet
   • Gizli özne: 2 adet

🔍 DETAYLI BULGULAR

🔤 "O çok yorgundu" cümlesinde 'O' zamiri 'Ali' kelimesine atıf yapmaktadır.

👻 "Hemen uyudu" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Ali' olarak tespit edilmiştir.

👻 "Sonucu öğrenmek için bilgisayarı açtılar" cümlesinde gizli özne (boş artgönderim) vardır. Öncül kelime 'Merve ve Cem' olarak tespit edilmiştir.



## 7. Değerlendirme ve Gradio Arayüzü


In [ ]:
# Örnek değerlendirme şablonu (DPATransLLM + ZARTR -> 50+50)
# Beklenen kolonlar: sentence, pronoun_gold, zero_gold, combined_gold
# dpa_df = pd.read_csv('/path/dpatransllm_test.csv')
# zartr_df = pd.read_csv('/path/zartr_test.csv')
# eval_df = resolver.build_integrated_eval_set(dpa_df, zartr_df, per_dataset=50)
# metrics = resolver.evaluate(eval_df)
# print(metrics)


In [ ]:
def create_html(result: Dict) -> str:
    """HTML çıktı"""
    html = '<div style="font-family: Arial, sans-serif; padding: 15px;">'

    # Özet
    html += '<div style="background: #e8f4f8; padding: 15px; border-radius: 8px; margin-bottom: 20px;">'
    html += f'<strong>📈 Özet:</strong> {len(result["sentences"])} cümle | '
    html += f'<span style="color: #27ae60;">🔤 {result["pronoun_count"]} zamir</span> | '
    html += f'<span style="color: #e67e22;">👻 {result["zero_count"]} gizli özne</span>'
    html += '</div>'

    # Her cümle için
    for sent_id, sentence in enumerate(result['sentences']):
        sent_findings = [f for f in result['findings'] if f.sentence_id == sent_id]

        # Cümle kutusu
        bg = '#f0f9ff' if not sent_findings else ('#d4edda' if any(f.finding_type == AnaphoraType.PRONOUN for f in sent_findings) else '#fff3cd')

        html += f'<div style="background: {bg}; padding: 15px; margin: 10px 0; border-radius: 8px; border-left: 4px solid #3498db;">'
        html += f'<strong>[{sent_id + 1}]</strong> {sentence}'

        for f in sent_findings:
            if f.finding_type == AnaphoraType.PRONOUN:
                html += f'<div style="margin-top: 10px; padding: 10px; background: #27ae60; color: white; border-radius: 5px;">'
                html += f'🔤 <strong>Zamir:</strong> "<em>{f.trigger_word}</em>" → "<em>{f.antecedent}</em>" kelimesine atıf yapmaktadır.'
                html += '</div>'
            else:
                html += f'<div style="margin-top: 10px; padding: 10px; background: #e67e22; color: white; border-radius: 5px;">'
                html += f'👻 <strong>Gizli Özne:</strong> Bu cümlede özne eksiltisi var. Öncül: "<em>{f.antecedent}</em>"'
                html += '</div>'

        if not sent_findings:
            html += '<div style="margin-top: 8px; color: #6c757d; font-size: 0.9em;">✓ Artgönderim yok</div>'

        html += '</div>'

    html += '</div>'
    return html


def analyze_text(text: str):
    """Gradio analiz fonksiyonu"""
    if not text.strip():
        return "<p>Lütfen metin girin.</p>", "Metin girilmedi."

    result = resolver.analyze(text)

    # Basit metin çıktı
    simple = []
    for f in result['findings']:
        simple.append(f.explain())

    text_output = "\n\n".join(simple) if simple else "Artgönderim bulunamadı."

    return create_html(result), text_output


EXAMPLES = [
    ["Ali dün eve geç geldi. O çok yorgundu. Hemen uyudu."],
    ["Merve ve Cem hemen odaya girdiler. Sonucu öğrenmek için bilgisayarı açtılar."],
    ["Ayşe kitabı aldı. Onu çantasına koydu. Eve gidince okumaya başladı."],
    ["Öğretmen sınıfa girdi. O bugün çok mutluydu. Öğrencilere güzel haberler verdi."],
    ["Çocuklar parkta oynadılar. Onlar çok eğlendiler. Akşama kadar koşturdular."]
]

print("✅ Gradio fonksiyonları hazır")

✅ Gradio fonksiyonları hazır


In [ ]:
with gr.Blocks(title="Türkçe Artgönderim", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🇹🇷 Türkçe Artgönderim Çözümleyici

    **Zamir artgönderimi** ve **gizli özne (boş artgönderim)** tespiti yapar.

    ---
    """)

    input_text = gr.Textbox(
        label="📝 Metin",
        placeholder="Türkçe metni buraya yazın...",
        lines=3
    )

    with gr.Row():
        analyze_btn = gr.Button("🔍 Analiz Et", variant="primary")
        clear_btn = gr.Button("🗑️ Temizle")

    gr.Markdown("### 📚 Örnekler")
    gr.Examples(examples=EXAMPLES, inputs=input_text)

    with gr.Tabs():
        with gr.TabItem("🎨 Görsel"):
            output_html = gr.HTML()
        with gr.TabItem("📝 Metin"):
            output_text = gr.Textbox(label="Açıklamalar", lines=8)

    gr.Markdown("""
    ---
    | Tür | Açıklama |
    |-----|----------|
    | 🔤 Zamir | O, Bu, Onlar gibi zamirler |
    | 👻 Gizli Özne | Yazılmayan özne |
    """)

    analyze_btn.click(analyze_text, inputs=[input_text], outputs=[output_html, output_text])
    clear_btn.click(lambda: ("", "", ""), outputs=[input_text, output_html, output_text])
    input_text.submit(analyze_text, inputs=[input_text], outputs=[output_html, output_text])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73c69313f613eacfe0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://73c69313f613eacfe0.gradio.live
